## Clustering con SparkML


## Objetivos

Después de completar este laboratorio serás capaz de:

 - Usar PySpark para conectar a un clúster de Spark.
 - Crear una sesión de Spark.
 - Leer un archivo CSV en un DataFrame.
 - Usar el algoritmo KMeans para agrupar los datos.
 - Detener la sesión de Spark.
 
 
 


## Conjuntos de datos

En este laboratorio utilizarás el/los siguiente(s) conjunto(s) de datos:

 - Versión modificada del conjunto de datos Wholesale customers. El conjunto original está en https://archive.ics.uci.edu/ml/datasets/Wholesale+customers 
 - Conjunto de datos Seeds. Disponible en https://archive.ics.uci.edu/ml/datasets/seeds


----


## Configuración


En este laboratorio usaremos las siguientes librerías:

*   [`PySpark`](https://spark.apache.org/docs/latest/api/python/index.html?utm_medium=Exinfluencer&utm_source=Exinfluencer&utm_content=000026UJ&utm_term=10006555&utm_id=NA-SkillsNetwork-Channel-SkillsNetworkCoursesIBMSkillsNetworkBD0231ENCoursera2789-2023-01-01) para conectar al clúster de Spark


### Instalación de las librerías necesarias

El clúster de Spark está preinstalado en el entorno de Skills Network Labs. Sin embargo, necesitas librerías como pyspark y findspark para conectarte a este clúster.

Si deseas descargar este cuaderno Jupyter y ejecutarlo en tu ordenador, sigue las instrucciones <a href="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMSkillsNetwork-BD0231EN-Coursera/labs/Connecting_to_spark_cluster_using_Skills_Network_labs.ipynb">aquí</a>.



Las siguientes librerías __no__ están preinstaladas en el entorno de Skills Network Labs. __Debes ejecutar la siguiente celda__ para instalarlas:


In [1]:
!pip install pyspark==3.1.2 -q
!pip install findspark -q

### Importar las librerías necesarias

_Recomendamos importar todas las librerías necesarias en un solo lugar (aquí):_


In [2]:
# You can also use this section to suppress warnings generated by your code:
def warn(*args, **kwargs):
    pass
import warnings
warnings.warn = warn
warnings.filterwarnings('ignore')

# FindSpark simplifies the process of using Apache Spark with Python

import findspark
findspark.init()

#import functions/Classes for sparkml

from pyspark.ml.clustering import KMeans
from pyspark.ml.feature import VectorAssembler

from pyspark.sql import SparkSession


## Ejemplos


## Tarea 1 - Crear una sesión de Spark


In [3]:
#Create SparkSession
#Ignore any warnings by SparkSession command

spark = SparkSession.builder.appName("Clustering using SparkML").getOrCreate()

26/02/15 03:54:44 WARN util.NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/02/15 03:54:47 WARN util.Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


## Tarea 2 - Cargar los datos de un CSV en un DataFrame


Descargar el archivo de datos


In [4]:
!wget https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-BD0231EN-SkillsNetwork/datasets/customers.csv


--2026-02-15 03:54:56--  https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-BD0231EN-SkillsNetwork/datasets/customers.csv
Resolving cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud (cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud)... 169.63.118.104, 169.63.118.104
Connecting to cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud (cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud)|169.63.118.104|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 8909 (8.7K) [text/csv]
Saving to: ‘customers.csv’

customers.csv       100%[===================>]   8.70K  --.-KB/s    in 0s      

2026-02-15 03:54:56 (42.6 MB/s) - ‘customers.csv’ saved [8909/8909]



Cargar el conjunto de datos en el DataFrame de Spark


In [5]:
# using the spark.read.csv function we load the data into a dataframe.
# the header = True mentions that there is a header row in out csv file
# the inferSchema = True, tells spark to automatically find out the data types of the columns.

# Load customers dataset
customer_data = spark.read.csv("customers.csv", header=True, inferSchema=True)


Imprimir el esquema del conjunto de datos


In [ ]:
# Each row in this dataset is about a customer. The columns indicate the orders placed
# by a customer for Fresh_food, Milk, Grocery and Frozen_Food

In [6]:
customer_data.printSchema()

root
 |-- Fresh_Food: integer (nullable = true)
 |-- Milk: integer (nullable = true)
 |-- Grocery: integer (nullable = true)
 |-- Frozen_Food: integer (nullable = true)



Mostrar las primeras 5 filas del conjunto de datos


In [7]:
customer_data.show(n=5, truncate=False)

+----------+----+-------+-----------+
|Fresh_Food|Milk|Grocery|Frozen_Food|
+----------+----+-------+-----------+
|12669     |9656|7561   |214        |
|7057      |9810|9568   |1762       |
|6353      |8808|7684   |2405       |
|13265     |1196|4221   |6404       |
|22615     |5410|7198   |3915       |
+----------+----+-------+-----------+
only showing top 5 rows



## Tarea 3 - Crear un vector de características


In [8]:
# Assemble the features into a single vector column
feature_cols = ['Fresh_Food', 'Milk', 'Grocery', 'Frozen_Food']
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")
customer_transformed_data = assembler.transform(customer_data)


Debes indicar al algoritmo KMeans cuántos clústeres crear con tus datos


In [9]:
number_of_clusters = 3

## Tarea 4 - Crear un modelo de clustering


Crear un modelo de clustering KMeans


In [10]:
kmeans = KMeans(k = number_of_clusters)


Entrenar/ajustar el modelo con el conjunto de datos<br>


In [11]:
model = kmeans.fit(customer_transformed_data)


26/02/15 03:56:31 WARN netlib.BLAS: Failed to load implementation from: com.github.fommil.netlib.NativeSystemBLAS
26/02/15 03:56:31 WARN netlib.BLAS: Failed to load implementation from: com.github.fommil.netlib.NativeRefBLAS


## Tarea 5 - Mostrar los detalles de los clústeres


Tu modelo ya está entrenado. Es el momento de evaluarlo.


In [12]:
# Make predictions on the dataset
predictions = model.transform(customer_transformed_data)

In [13]:
# Display the results
predictions.show(5)

+----------+----+-------+-----------+--------------------+----------+
|Fresh_Food|Milk|Grocery|Frozen_Food|            features|prediction|
+----------+----+-------+-----------+--------------------+----------+
|     12669|9656|   7561|        214|[12669.0,9656.0,7...|         0|
|      7057|9810|   9568|       1762|[7057.0,9810.0,95...|         0|
|      6353|8808|   7684|       2405|[6353.0,8808.0,76...|         0|
|     13265|1196|   4221|       6404|[13265.0,1196.0,4...|         0|
|     22615|5410|   7198|       3915|[22615.0,5410.0,7...|         2|
+----------+----+-------+-----------+--------------------+----------+
only showing top 5 rows



Mostrar cuántos clientes hay en cada clúster.


In [14]:
predictions.groupBy('prediction').count().show()

[Stage 43:=====================================================>  (72 + 3) / 75]

+----------+-----+
|prediction|count|
+----------+-----+
|         1|   49|
|         2|   60|
|         0|  331|
+----------+-----+



In [15]:
#stop spark session
spark.stop()

# Ejercicios


### Ejercicio 1 - Crear una sesión de Spark


Crear la SparkSession con el nombre de aplicación "Seed Clustering"


In [16]:
spark = SparkSession.builder.appName("Seed Clustering").getOrCreate()

26/02/15 04:07:54 WARN util.Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


<details>
    <summary>Haz clic aquí para ver una pista</summary>
    
Usa SparkSession.builder

</details>


<details>
    <summary>Haz clic aquí para ver la solución</summary>

```python
spark = SparkSession.builder.appName("Seed Clustering").getOrCreate()
```

</details>


### Ejercicio 2 - Cargar los datos de un CSV en un DataFrame


In [17]:
#download seed dataset
!wget https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-BD0231EN-SkillsNetwork/datasets/seeds.csv


--2026-02-15 04:07:58--  https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-BD0231EN-SkillsNetwork/datasets/seeds.csv
Resolving cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud (cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud)... 169.63.118.104, 169.63.118.104
Connecting to cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud (cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud)|169.63.118.104|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 8973 (8.8K) [text/csv]
Saving to: ‘seeds.csv’

seeds.csv           100%[===================>]   8.76K  --.-KB/s    in 0s      

2026-02-15 04:07:58 (38.6 MB/s) - ‘seeds.csv’ saved [8973/8973]



Cargar el conjunto de datos de semillas


In [18]:

seed_data = spark.read.csv("seeds.csv", header=True, inferSchema=True)


<details>
    <summary>Haz clic aquí para ver una pista</summary>
    
Usa spark.read.csv

</details>


<details>
    <summary>Haz clic aquí para ver la solución</summary>

```python
seed_data = spark.read.csv("seeds.csv", header=True, inferSchema=True)
```

</details>


Imprimir el esquema del conjunto de datos


In [19]:
seed_data.printSchema()

root
 |-- area: double (nullable = true)
 |-- perimeter: double (nullable = true)
 |-- compactness: double (nullable = true)
 |-- length of kernel: double (nullable = true)
 |-- width of kernel: double (nullable = true)
 |-- asymmetry coefficient: double (nullable = true)
 |-- length of kernel groove: double (nullable = true)



Mostrar las primeras 5 filas del conjunto de datos


In [20]:
seed_data.show(n=5, truncate=False, vertical=True)

-RECORD 0-------------------------
 area                    | 15.26  
 perimeter               | 14.84  
 compactness             | 0.871  
 length of kernel        | 5.763  
 width of kernel         | 3.312  
 asymmetry coefficient   | 2.221  
 length of kernel groove | 5.22   
-RECORD 1-------------------------
 area                    | 14.88  
 perimeter               | 14.57  
 compactness             | 0.8811 
 length of kernel        | 5.554  
 width of kernel         | 3.333  
 asymmetry coefficient   | 1.018  
 length of kernel groove | 4.956  
-RECORD 2-------------------------
 area                    | 14.29  
 perimeter               | 14.09  
 compactness             | 0.905  
 length of kernel        | 5.291  
 width of kernel         | 3.337  
 asymmetry coefficient   | 2.699  
 length of kernel groove | 4.825  
-RECORD 3-------------------------
 area                    | 13.84  
 perimeter               | 13.94  
 compactness             | 0.8955 
 length of kernel   

### Ejercicio 3 - Crear un vector de características


Ensamblar todas las columnas en un único vector


In [21]:
feature_cols = ['area',
 'perimeter',
 'compactness',
 'length of kernel',
 'width of kernel',
 'asymmetry coefficient',
 'length of kernel groove']
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")
seed_transformed_data = assembler.transform(seed_data)


<details>
    <summary>Haz clic aquí para ver una pista</summary>
    
Consulta la tarea 3
</details>


<details>
    <summary>Haz clic aquí para ver la solución</summary>

```python
feature_cols = ['area',
 'perimeter',
 'compactness',
 'length of kernel',
 'width of kernel',
 'asymmetry coefficient',
 'length of kernel groove']

assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")
seed_transformed_data = assembler.transform(seed_data)

```

</details>


### Ejercicio 4 - Crear un modelo de clustering


Crear 7 clústeres


In [22]:
number_of_clusters = 3
kmeans = KMeans(k = number_of_clusters)
model = kmeans.fit(seed_transformed_data)

<details>
    <summary>Haz clic aquí para ver una pista</summary>
    
Usa el método kmeans.fit()
</details>


<details>
    <summary>Haz clic aquí para ver la solución</summary>

```python
number_of_clusters = 3
kmeans = KMeans(k = number_of_clusters)
model = kmeans.fit(seed_transformed_data)

```

</details>


### Ejercicio 5 - Mostrar los detalles de los clústeres


In [ ]:
predictions = model.transform(seed_transformed_data)

<details>
    <summary>Haz clic aquí para ver una pista</summary>
    
Usa el método transform()
</details>


<details>
    <summary>Haz clic aquí para ver la solución</summary>

```python
predictions = model.transform(seed_transformed_data)
```

</details>


In [ ]:
predictions.show(n=5, truncate=False, vertical=True)

In [ ]:
predictions.groupBy('prediction').count().show()

In [ ]:
#stop spark session
spark.stop()

## Autores


[Ramesh Sannareddy](https://www.linkedin.com/in/rsannareddy/?utm_medium=Exinfluencer&utm_source=Exinfluencer&utm_content=000026UJ&utm_term=10006555&utm_id=NA-SkillsNetwork-Channel-SkillsNetworkCoursesIBMBD0231ENSkillsNetwork866-2023-01-01)


### Otros colaboradores


[Geronimo Saldaña](https://www.linkedin.com/in/geronimo-saldaña-espinal-b253821a7/)


Copyright © 2023 IBM Corporation. Todos los derechos reservados.


<!--## Change Log
-->


<!--
|Date (YYYY-MM-DD)|Version|Changed By|Change Description|
|-|-|-|-|
|2023-05-01|0.1|Ramesh Sannareddy|Initial Version Created|
-->
